In [ ]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
try:
    DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    DIR = os.getcwd()
os.chdir(DIR)

In [ ]:
torch.manual_seed(67)

In [ ]:
FEATURES = ["diam_min_m", "diam_max_m", "vel_km_h", "miss_km"]
TARGET = "magnitude"
RANDOM_SEED = 67
TEST_FRAC = 0.2
HIDDEN = 24
EPOCHS = 40
BATCH = 256
LR = 1e-3
N_PLANETS = 6

In [ ]:
df = pd.read_csv("../NEODodge/neo.csv")
df = df.loc[df.groupby("name")["miss_km"].idxmin()].reset_index(drop=True)
print("Asteroides únicos:", len(df))

In [ ]:
top_planets = df["orbiting"].value_counts().head(N_PLANETS).index.tolist()

In [ ]:
def planet_oht(s):
    row = [1.0 if s == p else 0.0 for p in top_planets]
    row.append(0.0 if s in top_planets else 1.0)
    return row

In [ ]:
X_oht = np.array([planet_oht(s) for s in df["orbiting"]], dtype=np.float32)

In [ ]:
Xl = df[FEATURES].apply(np.log1p).to_numpy(dtype=np.float32)
y = df[TARGET].to_numpy(dtype=np.float32)

In [ ]:
# Split sin stratify (target continuo)
idx = np.arange(len(Xl))
itr, itest = train_test_split(idx, test_size=TEST_FRAC, random_state=RANDOM_SEED)
xtr, xtest = Xl[itr], Xl[itest]
otr, otest = X_oht[itr], X_oht[itest]
ytr, ytest = y[itr], y[itest]

In [ ]:
# z-score con mu/sigma del TRAIN, solo sobre las 4 features
mu = xtr.mean(0)
sigma = xtr.std(0) + 1e-8
xtr = (xtr - mu) / sigma
xtest = (xtest - mu) / sigma

In [ ]:
def make_model(n_in):
    return nn.Sequential(
        nn.Linear(n_in, HIDDEN),
        nn.ReLU(),
        nn.Linear(HIDDEN, 1),
    )

In [ ]:
def train_model(Xtr, Ytr, Xva, Yva, Otr=None, Ova=None):
    Xtr_a = Xtr if Otr is None else np.hstack([Xtr, Otr])
    Xva_a = Xva if Ova is None else np.hstack([Xva, Ova])

    model = make_model(Xtr_a.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    lossf = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr_a, dtype=torch.float32)
    Ytr_t = torch.tensor(Ytr, dtype=torch.float32)
    Xva_t = torch.tensor(Xva_a, dtype=torch.float32)
    ds = torch.utils.data.TensorDataset(Xtr_t, Ytr_t)
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True)

    for ep in range(EPOCHS):
        model.train()
        tot, n = 0.0, 0
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb).squeeze(1)
            loss = lossf(out, yb)
            loss.backward()
            opt.step()
            tot += loss.item() * xb.size(0)
            n += xb.size(0)
        if (ep + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                pred = model(Xva_t).numpy().ravel()
            print(f"  ep {ep+1:3d} | train MSE {tot/n:.4f} | val MAE {np.mean(np.abs(pred - Yva)):.3f}")

    model.eval()
    with torch.no_grad():
        pred = model(Xva_t).numpy().ravel()
    mse = mean_squared_error(Yva, pred)
    met = {
        "mse": float(mse),
        "rmse": float(np.sqrt(mse)),
        "mae": float(mean_absolute_error(Yva, pred)),
        "r2": float(r2_score(Yva, pred)),
    }
    return model, met, pred

In [ ]:
print("\nModelo A: solo geometria (4 features)")
mA, metA, predA = train_model(xtr, ytr, xtest, ytest)
print("A ->", metA)

In [ ]:
print("\nModelo B: geometria + orbiting (one-hot)")
mB, metB, predB = train_model(xtr, ytr, xtest, ytest, otr, otest)
print("B ->", metB)

In [ ]:
with_planet = metB["mae"] <= metA["mae"]
best, met, pred = (mB, metB, predB) if with_planet else (mA, metA, predA)
print(f"\nGanador: modelo {'B (+orbiting)' if with_planet else 'A'} | "
      f"val MAE {met['mae']:.3f} | val R2 {met['r2']:.3f}")

In [ ]:
n_in = best[0].in_features
torch.save(best.state_dict(), "mag_model.pt")
with open("mag_stats.json", "w") as f:
    json.dump({
        "features": FEATURES,
        "mu": mu.tolist(),
        "sigma": sigma.tolist(),
        "planets": top_planets + ["other"],
        "with_orbiting": bool(with_planet),
        "n_in": n_in,
        "hidden": HIDDEN,
    }, f, indent=2)

In [ ]:
freq = df["orbiting"].value_counts()
counts = {p: int(freq.get(p, 0)) for p in (top_planets + ["other"])}
counts["other"] = int(freq.drop(top_planets).sum())
with open("planets.json", "w") as f:
    json.dump(counts, f, indent=2)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(ytest, pred, s=4, alpha=0.4)
lim = [min(ytest.min(), pred.min()), max(ytest.max(), pred.max())]
plt.plot(lim, lim, "r--")
plt.xlabel("magnitude real")
plt.ylabel("magnitude predicha")
plt.title(f"MAE {met['mae']:.3f} | R2 {met['r2']:.3f}")
plt.tight_layout()
plt.show()